In [1]:
import torch
from torch.utils.data import TensorDataset, DataLoader

sentences = ["what is statquest <EOS> awesome",
             "statquest is what <EOS> awesome",
             "squatch eats pizza <EOS> yum",    # the usage of position encoding
             "pizza eats squatch <EOS> yikes"]

token_to_id = {'what': 0,
               'is': 1,
               'statquest': 2,
               'awesome': 3,
               "squatch": 4,
               "eats": 5,
               "pizza": 6,
               "yum": 7,
               "yikes": 8,
               '<EOS>': 9} # <EOS> = end of sequence

id_to_token = dict(map(reversed, token_to_id.items()))

inputs = torch.tensor([[token_to_id[token] for token in sentence.split()]
                       for sentence in sentences])
# what is statquest <EOS> awesome
# statquest is what <EOS> awesome

labels = torch.tensor([[token_to_id[token] for token in sentence.split()[1:]] + [token_to_id['<EOS>']]
                       for sentence in sentences])
# is statquest <EOS> awesome <EOS>
# is what <EOS> awesome <EOS>

dataset = TensorDataset(inputs, labels)
dataloader = DataLoader(dataset)

In [15]:
from importlib import reload
from src import transformer
reload(transformer)
from src.transformer import DecoderOnlyTransformer

import lightning as L

max_length = 6
model = DecoderOnlyTransformer(num_tokens=len(token_to_id), d_model=2, max_len=max_length)

In [18]:
### Check

eos_id = token_to_id['<EOS>']

model_input = ["what is statquest <EOS>",
               "statquest is what <EOS>",
               "squatch eats pizza <EOS>",
               "pizza eats squatch <EOS>"]

model_input = torch.tensor([[token_to_id[token] for token in sentence.split()]
                           for sentence in model_input])
input_length = model_input.size(dim=1)
predicted_ids = torch.tensor([])
for _ in range(input_length, max_length):
  predictions = model(model_input)
  predicted_id = torch.argmax(predictions[:, -1:], dim=-1)
  # predicted_id: [batch_size, 1]
  predicted_ids = torch.cat([predicted_ids, predicted_id], dim=-1)

  model_input = torch.cat([model_input, predicted_id], dim=-1)

print("Predicted Tokens:")
for batch in predicted_ids:
  print(" ".join([id_to_token[id.item()] for id in batch]))

Predicted Tokens:
awesome <EOS>
awesome <EOS>
yum <EOS>
yikes <EOS>


In [17]:
### Training

trainer = L.Trainer(max_epochs=30)
trainer.fit(model, train_dataloaders=dataloader)

INFO: GPU available: True (mps), used: True
2025-03-05 14:29:16,886 - lightning.pytorch.utilities.rank_zero - INFO - GPU available: True (mps), used: True
INFO: TPU available: False, using: 0 TPU cores
2025-03-05 14:29:16,888 - lightning.pytorch.utilities.rank_zero - INFO - TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
2025-03-05 14:29:16,889 - lightning.pytorch.utilities.rank_zero - INFO - HPU available: False, using: 0 HPUs
INFO: 
  | Name           | Type             | Params | Mode 
------------------------------------------------------------
0 | we             | Embedding        | 20     | train
1 | pe             | PositionEncoding | 0      | train
2 | self_attention | Attention        | 12     | train
3 | fc_layer       | Linear           | 30     | train
4 | criterion      | CrossEntropyLoss | 0      | train
------------------------------------------------------------
62        Trainable params
0         Non-trainable params
62        Total 

Training: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=30` reached.
2025-03-05 14:29:23,104 - lightning.pytorch.utilities.rank_zero - INFO - `Trainer.fit` stopped: `max_epochs=30` reached.
